# Agent CSV generator — GAMECHAR validation games

Generates one 265-column aligned agent CSV per game for the six validation games.

| # | Game | Algorithm | Source |
|---|---|---|---|
| 1 | Breakout | DQN | sb3 pretrained |
| 2 | Space Invaders | DQN | sb3 pretrained |
| 3 | Seaquest | DQN | sb3 pretrained |
| 4 | Freeway | PPO | trained here |
| 5 | Asterix | PPO | trained here |
| 6 | Skiing | PPO | trained here (expected score ~-30000) |

**Output:** `agent_test_data/<game>/<algo>_<game>_<N>ep.csv`

**Runtime (CPU):** pretrained games ~2 min each; trained games 15-30 min each.
Set `TRAIN_TIMESTEPS = 500_000` and `N_EVAL_EPISODES = 5` for a quick test run.


## 0. Install dependencies

The `gym` (old OpenAI Gym) package is required alongside `gymnasium` because the pretrained sb3 DQN models were pickled with `gym` and will fail to load without it.

In [ ]:
import subprocess, sys

pkgs = [
    "gym==0.26.2",                         # needed to unpickle pretrained sb3 models
    "stable-baselines3[extra]>=2.0",
    "rl_zoo3",
    "huggingface-sb3",
    "ale-py>=0.9",
    "gymnasium[atari,accept-rom-license]",
]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=False)
print("Done. Restart the kernel now if this was a fresh install.")


## 1. Imports and configuration

In [ ]:
from __future__ import annotations
import datetime, warnings
from pathlib import Path

import ale_py
import gymnasium as gym
import numpy as np
import pandas as pd
from stable_baselines3 import DQN, PPO
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack, VecTransposeImage
from huggingface_sb3 import load_from_hub

warnings.filterwarnings("ignore")

# Register ALE environments with Gymnasium (required in gymnasium >= 0.26)
gym.register_envs(ale_py)

# ── settings ───────────────────────────────────────────────────────────────
OUT_ROOT         = Path("agent_test_data")
N_EVAL_EPISODES  = 30
MAX_STEPS_PER_EP = 10_000
EVAL_SEED        = 0
TRAIN_TIMESTEPS  = 1_000_000
TRAIN_SEED       = 0

BASE_COLS = ["run_ts", "episode", "step", "action", "reward", "done",
             "episode_return", "lives_pre", "lives_post"]
RAM_PRE  = [f"ram_pre_{i}"  for i in range(128)]
RAM_POST = [f"ram_post_{i}" for i in range(128)]
ALL_COLS = BASE_COLS + RAM_PRE + RAM_POST   # 265

print(f"Gymnasium : {gym.__version__}  |  ale-py : {ale_py.__version__}")
print(f"Output    : {OUT_ROOT.resolve()}")
print(f"Episodes  : {N_EVAL_EPISODES}  |  Max steps/ep : {MAX_STEPS_PER_EP}")
print(f"Train steps (Freeway/Asterix/Skiing) : {TRAIN_TIMESTEPS:,}")


## 2. Shared helpers

In [ ]:
def make_env(env_id: str, seed: int = EVAL_SEED, n_envs: int = 1) -> VecFrameStack:
    vec = make_atari_env(env_id, n_envs=n_envs, seed=seed)
    return VecFrameStack(vec, n_stack=4)


def get_ale(env):
    """Recursively unwrap to the ALE interface."""
    if hasattr(env, "envs"):  return get_ale(env.envs[0])
    if hasattr(env, "env"):   return get_ale(env.env)
    if hasattr(env, "ale"):   return env.ale
    raise RuntimeError(f"ALE not found in env chain: {type(env)}")


def run_and_log(model, env_id: str, game: str, algo: str,
                n_episodes: int = N_EVAL_EPISODES,
                max_steps: int = MAX_STEPS_PER_EP,
                seed: int = EVAL_SEED) -> Path:
    """Evaluate model for n_episodes and write 265-column CSV."""
    out_dir  = OUT_ROOT / game
    out_dir.mkdir(parents=True, exist_ok=True)
    run_ts   = datetime.datetime.utcnow().strftime("%Y-%m-%dT%H-%M-%S")
    out_path = out_dir / f"{algo}_{game}_{n_episodes}ep.csv"

    eval_env = make_atari_env(env_id, n_envs=1, seed=seed)
    eval_env = VecFrameStack(eval_env, n_stack=4)
    ale      = get_ale(eval_env)

    rows, ep, ep_step, ep_return, ep_num, lives_pre = [], 0, 0, 0.0, 1, None
    obs = eval_env.reset()

    while ep < n_episodes:
        ram_pre = ale.getRAM().copy()
        if lives_pre is None:
            lives_pre = int(ale.lives())

        action_arr, _ = model.predict(obs, deterministic=True)
        action = int(action_arr[0])
        obs, reward_arr, done_arr, _ = eval_env.step(action_arr)
        reward     = float(reward_arr[0])
        done       = bool(done_arr[0])
        ram_post   = ale.getRAM().copy()
        lives_post = int(ale.lives())
        ep_return += reward
        ep_step   += 1

        row = {"run_ts": run_ts, "episode": ep_num, "step": ep_step,
               "action": action, "reward": reward,
               "done": int(done or ep_step >= max_steps),
               "episode_return": ep_return,
               "lives_pre": lives_pre, "lives_post": lives_post}
        for i in range(128):
            row[f"ram_pre_{i}"]  = int(ram_pre[i])
            row[f"ram_post_{i}"] = int(ram_post[i])
        rows.append(row)
        lives_pre = lives_post

        if done or ep_step >= max_steps:
            ep       += 1
            print(f"  ep {ep:3d}/{n_episodes}  return={ep_return:10.1f}  steps={ep_step}", end="\r")
            ep_return = 0.0; ep_step = 0; ep_num += 1; lives_pre = None
            obs = eval_env.reset()

    eval_env.close()
    print()
    df = pd.DataFrame(rows)[ALL_COLS]
    df.to_csv(out_path, index=False)
    print(f"  Saved {len(df):,} rows → {out_path}")
    return out_path


def quick_validate(path: Path):
    df      = pd.read_csv(path)
    nc      = len(df.columns)
    ne      = df["episode"].nunique()
    mr      = df.groupby("episode")["episode_return"].last().mean()
    acts_ok = df["action"].between(0, 17).all()
    cum_ok  = (df.groupby("episode")
                 .apply(lambda g: abs(g["reward"].sum() - g["episode_return"].iloc[-1]) < 0.01)
                 .all())
    status  = "OK" if (nc == 265 and acts_ok) else "CHECK"
    print(f"  [{status}] cols={nc}/265  eps={ne}  mean_return={mr:.1f}  "
          f"actions_ok={acts_ok}  cumsum_ok={cum_ok}")


# Custom objects needed to load old sb3 DQN models.
# optimize_memory_usage + handle_timeout_termination conflict was allowed in
# older sb3 but raises ValueError in current sb3 — override both to False.
DQN_CUSTOM = {
    "learning_rate": 0.0,
    "lr_schedule": lambda _: 0.0,
    "exploration_schedule": lambda _: 0.01,
    "optimize_memory_usage": False,
    "handle_timeout_termination": False,
}

def load_dqn(repo_id: str, filename: str, env_id: str) -> DQN:
    print(f"Downloading {repo_id} ...")
    ckpt  = load_from_hub(repo_id=repo_id, filename=filename)
    env   = make_env(env_id)
    model = DQN.load(ckpt, env=env, custom_objects=DQN_CUSTOM)
    env.close()
    print("  Loaded.")
    return model


def load_or_train_ppo(env_id: str, game: str, ckpt_override=None) -> PPO:
    if ckpt_override is not None:
        p   = Path(ckpt_override)
        tmp = make_env(env_id)
        m   = PPO.load(str(p), env=tmp)
        tmp.close()
        print(f"Loaded checkpoint: {p}")
        return m
    print(f"Training PPO on {env_id} ({TRAIN_TIMESTEPS:,} steps) ...")
    train_env = make_atari_env(env_id, n_envs=8, seed=TRAIN_SEED)
    train_env = VecFrameStack(train_env, n_stack=4)
    train_env = VecTransposeImage(train_env)
    model = PPO("CnnPolicy", train_env, seed=TRAIN_SEED,
                learning_rate=2.5e-4, n_steps=128, batch_size=256,
                n_epochs=4, gamma=0.99, gae_lambda=0.95,
                clip_range=0.1, ent_coef=0.01, vf_coef=0.5,
                max_grad_norm=0.5, verbose=0)
    model.learn(total_timesteps=TRAIN_TIMESTEPS)
    ckpt = OUT_ROOT / game / f"ppo_{game}_trained.zip"
    ckpt.parent.mkdir(parents=True, exist_ok=True)
    model.save(str(ckpt))
    train_env.close()
    print(f"  Checkpoint saved → {ckpt}")
    return model


## 3. Breakout — DQN (sb3 pretrained)

In [ ]:
model = load_dqn("sb3/dqn-BreakoutNoFrameskip-v4", "dqn-BreakoutNoFrameskip-v4.zip", "BreakoutNoFrameskip-v4")
path  = run_and_log(model, "BreakoutNoFrameskip-v4", "breakout", "dqn")
quick_validate(path)
del model


## 4. Space Invaders — DQN (sb3 pretrained)

In [ ]:
model = load_dqn("sb3/dqn-SpaceInvadersNoFrameskip-v4", "dqn-SpaceInvadersNoFrameskip-v4.zip", "SpaceInvadersNoFrameskip-v4")
path  = run_and_log(model, "SpaceInvadersNoFrameskip-v4", "space_invaders", "dqn")
quick_validate(path)
del model


## 5. Seaquest — DQN (sb3 pretrained)

In [ ]:
model = load_dqn("sb3/dqn-SeaquestNoFrameskip-v4", "dqn-SeaquestNoFrameskip-v4.zip", "SeaquestNoFrameskip-v4")
path  = run_and_log(model, "SeaquestNoFrameskip-v4", "seaquest", "dqn")
quick_validate(path)
del model


## 6. Freeway — PPO (trained here)
PPO converges quickly (~15 min CPU). No sb3 model exists.
Set `FREEWAY_CKPT` to an existing `.zip` path to skip training.

In [ ]:
FREEWAY_CKPT = None   # e.g. "agent_test_data/freeway/ppo_freeway_trained.zip"

model = load_or_train_ppo("FreewayNoFrameskip-v4", "freeway", FREEWAY_CKPT)
path  = run_and_log(model, "FreewayNoFrameskip-v4", "freeway", "ppo")
quick_validate(path)
del model


## 7. Asterix — PPO (trained here)
Slightly harder than Freeway (~20-30 min CPU). No sb3 model exists.
Set `ASTERIX_CKPT` to an existing `.zip` path to skip training.

In [ ]:
ASTERIX_CKPT = None   # e.g. "agent_test_data/asterix/ppo_asterix_trained.zip"

model = load_or_train_ppo("AsterixNoFrameskip-v4", "asterix", ASTERIX_CKPT)
path  = run_and_log(model, "AsterixNoFrameskip-v4", "asterix", "ppo")
quick_validate(path)
del model


## 8. Skiing — PPO (trained here)
No pretrained model exists. Agent will score ~-30000 (always times out). Expected — the human-vs-agent divergence on Skiing is the point.
Set `SKIING_CKPT` to an existing `.zip` path to skip training.

In [ ]:
SKIING_CKPT = None   # e.g. "agent_test_data/skiing/ppo_skiing_trained.zip"

model = load_or_train_ppo("SkiingNoFrameskip-v4", "skiing", SKIING_CKPT)
path  = run_and_log(model, "SkiingNoFrameskip-v4", "skiing", "ppo")
quick_validate(path)
del model


## 9. Summary

In [ ]:
print(f"{'game':16s} {'algo':4s} {'cols':6s} {'eps':5s} {'mean_return':>12s} {'status'}")
print("-" * 58)
for game, algo in [("breakout","dqn"),("space_invaders","dqn"),
                   ("seaquest","dqn"),("freeway","ppo"),
                   ("asterix","ppo"),("skiing","ppo")]:
    files = list(Path("agent_test_data", game).glob(f"{algo}_{game}*.csv"))
    if not files:
        print(f"{game:16s} {algo:4s} {'--':6s} {'--':5s} {'MISSING':>12s}")
        continue
    df = pd.read_csv(files[0])
    nc = len(df.columns); ne = df["episode"].nunique()
    mr = df.groupby("episode")["episode_return"].last().mean()
    print(f"{game:16s} {algo:4s} {nc:<6d} {ne:<5d} {mr:>12.1f} {'OK' if nc==265 else 'BAD_COLS'}")
print()
print("Ready. Place CSVs in agent_test_data/<game>/ and human aligned CSVs")
print("in human_raw_test_data/<game>/, then run:")
print("  python behavioural_pipeline.py --games breakout space_invaders seaquest freeway asterix skiing")
